# NutriScan — fine-tune bake-off (Colab GPU)

Runs `ml/train_finetune.py` on a GPU for both backbones (EfficientNetV2-S and ViT-S), then compares them against the Day-6 baseline on the frozen test set.

**Before you start:**
1. Locally run `uv run python ml/pack_dataset.py` → produces `ml/data/nutriscan_dataset_v1.zip`.
2. Upload that zip to Google Drive at `MyDrive/nutriscan/nutriscan_dataset_v1.zip`.
3. Here in Colab: **Runtime → Change runtime type → GPU** (T4 is fine).
4. Run the cells top to bottom.

The baseline to beat is **Top-1 86.1% / Top-5 97.1%**.

In [ ]:
# Confirm a GPU is attached (else: Runtime -> Change runtime type -> GPU)
!nvidia-smi -L

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Paths — edit DATA_ZIP if you put the zip somewhere else in Drive.
DATA_ZIP = "/content/drive/MyDrive/nutriscan/nutriscan_dataset_v1.zip"
REPO = "https://github.com/Shivamchaubey14/nutriscan.git"
REPO_DIR = "/content/nutriscan"
OUT_DIR = "/content/drive/MyDrive/nutriscan/results"  # winners are copied back here

In [ ]:
# Clone the repo (public) — brings train_finetune.py + the committed manifests.
![ -d {REPO_DIR} ] || git clone --depth 1 {REPO} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Unzip the packed dataset into ml/data/raw/ (paths match the manifests).
import os

os.makedirs("ml/data/raw", exist_ok=True)
!unzip -q -o "{DATA_ZIP}" -d ml/data/raw
!echo 'images:' && find ml/data/raw -type f | wc -l

In [ ]:
# Colab already has CUDA torch/torchvision — just add timm + mlflow.
!pip -q install 'timm>=1.0.28' 'mlflow>=3.14'

In [ ]:
# Train EfficientNetV2-S (full fine-tune). ~15 epochs; adjust to taste.
!python ml/train_finetune.py --backbone effv2s --epochs 15 --batch-size 64 --num-workers 2

In [ ]:
# Train ViT-S on the same data + protocol.
!python ml/train_finetune.py --backbone vits --epochs 15 --batch-size 64 --num-workers 2

In [ ]:
# Compare candidates vs the baseline on the frozen test set.
import glob
import json

rows = []
for path in sorted(glob.glob("ml/metrics/*_v1.json")):
    with open(path) as f:
        m = json.load(f)
    rows.append((m["model"], m.get("test_top1"), m.get("test_top5")))
print(f"{'model':<45}{'top1':>8}{'top5':>8}")
for name, t1, t5 in rows:
    print(f"{name:<45}{t1:>8.4f}{t5:>8.4f}")
print("\nBaseline to beat: top1 0.8606 / top5 0.9707")

In [ ]:
# Copy the trained models + metrics back to Drive so you can pull them locally.
import glob
import os
import shutil

os.makedirs(OUT_DIR, exist_ok=True)
for path in glob.glob("ml/models/*_v1.pt") + glob.glob("ml/metrics/*_v1.json"):
    shutil.copy2(path, OUT_DIR)
print("copied to", OUT_DIR)
!ls -la {OUT_DIR}

## Next (Day 20 wrap-up, done locally)

1. Download the winning `<backbone>_v1.pt` + `<backbone>_v1.json` from `MyDrive/nutriscan/results` into `ml/models/` and `ml/metrics/`.
2. Commit the metrics JSON (the `.pt` stays git-ignored).
3. If the winner beats the 86.1% baseline, export it to ONNX (Day-8 style: `inference/export_onnx.py`) and wire it into the inference service.

MLflow runs are under `mlruns/` in the Colab session — download it too if you want the full history.